# Disease diagnosis RAG — end-to-end walkthrough

Educational pipeline: **ingest → retrieve → rerank** (BM25, k-NN, hybrid + RRF, cross-encoder rerank).

## Prerequisites

1. `.env` at project root with `OPENSEARCH_*` credentials
2. OpenSearch index bootstrapped:
   ```bash
   uv run python -m src.migrations.init_db upgrade
   uv run python -m src.migrations.migrate_ddxplus_index upgrade
   ```
3. Run from project root or `notebooks/` with the project venv:
   ```bash
   cd disease-diagnosis-rag-system
   uv sync
   ```

First BGE / reranker calls download models into `models/` (lazy singleton).

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "services" / "rag").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists(), (
    "Run this notebook from the repo root or notebooks/ folder"
)

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json

from src.db.vector_db.opensearch import get_opensearch_client
from src.services.inference.embeddings.service import TextEmbeddingService
from src.services.inference.reranker.service import RerankerService
from src.services.rag import RAGService, Retriever
from src.services.rag.ingest import Ingestion
from src.services.rag.preprocess import PreprocessPipeline
from src.services.rag.schemas import (
    Bm25RetrieveRequest,
    DiseaseDocument,
    HybridRetrieveRequest,
    RetrievalMode,
    RetrieveExperimentRequest,
    RetrieveResult,
    VectorRetrieveRequest,
)
from src.settings import settings

## Configuration

Non-secret defaults from `src/settings.py` and `.env`. Paths resolve against the project root.

In [ ]:
print("Index alias:", settings.RETRIEVE_INDEX_ALIAS)
print("Search pipeline:", settings.CURRENT_SEARCH_PIPELINE)
print("Retrieve top_k:", settings.RETRIEVE_TOP_K)
print("Rerank top_k:", settings.RERANK_TOP_K)
print("Embedding model path:", settings.embedding_model_path)
print("Reranker model path:", settings.reranker_model_path)
print("OpenSearch host:", settings.OPENSEARCH_HOST)

## Step 1 — Ingest sample documents

`DiseaseDocument` holds source fields; `Ingestion` normalizes symptoms, embeds `embed_text`, derives `keyword_text`, and bulk-upserts into the `diseases` alias.

Skip this step if your index is already populated (e.g. full DDXPlus load).

In [ ]:
records = [
    DiseaseDocument(
        doc_id="demo-1",
        disease="Influenza",
        symptoms=["fever", "cough"],
        severity=2,
        source="ddxplus",
        description="Example disease for notebook smoke test.",
    )
]

embed_service = TextEmbeddingService()
ingestion = Ingestion(embed_service)
count = ingestion.ingest(records)

print(f"Ingested {count} record(s) into alias '{settings.RETRIEVE_INDEX_ALIAS}'")

## Step 2 — Query preprocessing

Normalize symptom tokens and expand synonyms before search (`tired` → `fatigue`).

In [ ]:
raw_query = "I am tired, have fever and cough"
preprocess = PreprocessPipeline()
search_query = preprocess.preprocess_query(raw_query)

print("Raw:", raw_query)
print("Preprocessed:", search_query)

## Step 3 — Initialize retrieval services

Reuses the embedding service from ingest. First reranker call loads `bge-reranker-base`.

- **`Retriever`** — BM25 / k-NN / hybrid APIs and composable `rerank()`
- **`RAGService`** — production path: hybrid retrieve (top 20) → rerank (top 5)

In [ ]:
client = get_opensearch_client()
rerank_service = RerankerService()
preprocess = PreprocessPipeline()

retriever = Retriever(
    client=client,
    embed_service=embed_service,
    preprocess=preprocess,
)
retriever_with_rerank = Retriever(
    client=client,
    embed_service=embed_service,
    preprocess=preprocess,
    rerank_service=rerank_service,
)

QUERY = "fever cough fatigue"
TOP_K = 5

In [ ]:
def show_hits(result: RetrieveResult, limit: int = 10) -> None:
    print(f"took_ms={result.took_ms}  hits={len(result.hits)}")
    for hit in result.hits[:limit]:
        score = f"{hit.score:.4f}" if hit.score is not None else "n/a"
        print(
            f"{hit.rank:>2}. {hit.disease:<24} "
            f"score={score}  symptoms={hit.symptoms}"
        )

## Step 4a — BM25 (lexical)

`match` on `keyword_text` with `operator: or`.

In [ ]:
bm25_result = retriever.search_bm25(
    Bm25RetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== BM25 ===")
show_hits(bm25_result)

## Step 4b — k-NN (semantic)

BGE query embedding (384-dim, cosine) on the `embedding` field.

In [ ]:
vector_result = retriever.search_vector(
    VectorRetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== k-NN ===")
show_hits(vector_result)

## Step 4c — Hybrid + RRF (MVP default)

Single OpenSearch `hybrid` query fused server-side via the `hybrid-rrf` pipeline.

In [ ]:
hybrid_result = retriever.search_hybrid(
    HybridRetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== Hybrid + RRF ===")
show_hits(hybrid_result)

## Step 5 — Compare all retrieval modes

Same query through BM25, k-NN, and hybrid side by side.

In [ ]:
comparison = retriever.run_experiment(
    RetrieveExperimentRequest(query=QUERY, top_k=TOP_K)
)

for mode, mode_result in comparison.results.items():
    print(f"\n=== {mode.value.upper()} (total_hits={mode_result.total_hits}) ===")
    show_hits(mode_result, limit=3)

## Step 6 — Retrieve + rerank (production path)

Hybrid search returns up to `RETRIEVE_TOP_K` (20) hits. Cross-encoder reranking keeps `RERANK_TOP_K` (5).

Each hit exposes `passage_text` — symptom-first text for the reranker. Query preprocessing runs via the injected `PreprocessPipeline`.

In [ ]:
retrieved = retriever_with_rerank.retrieve(QUERY)
print(f"Before rerank: {len(retrieved.hits)} hits")
if retrieved.hits:
    print("Sample passage_text:", retrieved.hits[0].passage_text[:120], "...")

reranked = retriever_with_rerank.rerank(
    QUERY, retrieved, top_k=settings.RERANK_TOP_K
)
print("\n=== After rerank (cross-encoder scores) ===")
show_hits(reranked)

rag = RAGService(
    embed_service=embed_service,
    rerank_service=rerank_service,
)
result = rag.query(QUERY)
print("\n=== RAGService.query() ===")
show_hits(result)

## Optional — Inspect OpenSearch request body

Enable debug bodies on the experiment request to see the Query DSL sent to OpenSearch.

In [ ]:
debug = retriever.run_experiment(
    RetrieveExperimentRequest(
        query=QUERY,
        top_k=TOP_K,
        include_opensearch_body=True,
    )
)

hybrid_debug = debug.results[RetrievalMode.HYBRID]
print(json.dumps(hybrid_debug.opensearch_body, indent=2)[:2000])

## Next steps

- **Full ingest** — load `data/kb/kb_ddxplus.json` via `RAGService.ingest()` (see `docs/ddxplus-index-mapping.md`)
- **Generate** — Qwen3 8B with reranked context
- **API** — FastAPI endpoint wrapping `RAGService.query()`